# PhageMatch-PK - Kleborate typing (self-contained)

Types 260 clinical *K. pneumoniae* genomes - Pakistani clinical K. pneumoniae:
ST, **K-locus (capsule)**, O-locus, AMR, virulence.

Accessions are embedded below - no uploads needed. Run cells in order.
**Cell 3 restarts the runtime on purpose**; resume at Cell 4 afterwards.


## Cell 1 - System tools

In [ ]:
%%bash
# No `set -e`: keep diagnostics visible even if one probe fails.
# mash is REQUIRED - Kleborate's species check shells out to the mash
# binary and aborts with 'could not find mash'. The pip package named
# 'mash' is a different project and does NOT provide the executable.
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y minimap2 ncbi-blast+ mash > /dev/null 2>&1
curl -sSL -o /usr/local/bin/datasets \
  https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets
chmod +x /usr/local/bin/datasets
echo '--- versions ---'
minimap2 --version || echo 'minimap2 MISSING'
datasets --version || echo 'datasets MISSING'
mash --version || echo 'mash MISSING'


## Cell 2 - Install Kleborate + Kaptive

**Kaptive is pinned to 3.2.2 on purpose.** Kleborate 3.2.4 declares an unpinned
`kaptive` dependency, but Kaptive 3.3.0 restructured its package and removed
`kaptive.database`, which Kleborate's capsule module imports. Installing latest
Kaptive gives `ModuleNotFoundError: No module named 'kaptive.database'`.
3.2.2 is the newest release that still provides it.


In [ ]:
import subprocess, sys

KAPTIVE_PIN = 'kaptive==3.2.2'   # see note above - do not unpin

def npver():
    r = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                       capture_output=True, text=True)
    return r.stdout.strip() or '(none)'

before = npver()
print('numpy before:', before)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', 'kleborate', KAPTIVE_PIN],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
if r.returncode != 0:
    print('PIP FAILED:', r.stderr[-3000:])
print('numpy after :', npver())


## Cell 3 - Restart runtime (expected; resume at Cell 4)

In [ ]:
import os
print('Restarting - this is expected. Continue at Cell 4.')
os.kill(os.getpid(), 9)


## Cell 4 - Verify install, detect CLI

In [ ]:
import subprocess

for mod in ['numpy', 'numba', 'kaptive', 'kleborate']:
    try:
        m = __import__(mod)
        print(f'{mod:<10}', getattr(m, '__version__', 'ok'))
    except Exception as e:
        print(f'{mod:<10} FAILED: {type(e).__name__}: {e}')

h = subprocess.run(['kleborate', '--help'], capture_output=True, text=True)
help_text = h.stdout + h.stderr
print(help_text[:4000])

for flag in ['--list-presets', '--list-modules']:
    r = subprocess.run(['kleborate', flag], capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if r.returncode == 0 and out:
        print(f'\n=== {flag} ===\n' + out[:2500])


## Cell 4b - Mount Google Drive (crash protection)

Colab free runtimes get reclaimed without warning, and anything in `/content`
dies with them. Typing takes ~50 min, so we keep the **results and per-chunk
progress markers on Drive** - a few MB. The 1.5 GB of genome FASTAs stay in
local scratch because re-downloading them costs only ~45 s.

Net effect: if the session dies, rerun Cells 1-7 and the typing resumes from
the last finished chunk instead of starting over.

**This cell asks for Google Drive access - approve it in the popup.**


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

PERSIST = Path('/content/drive/MyDrive/phagematch-pk')
PERSIST.mkdir(parents=True, exist_ok=True)
print('persisting results to:', PERSIST)

existing = sorted(PERSIST.rglob('chunk_*.done'))
print(f'chunks already finished from a previous session: {len(existing)}')


## Cell 5 - Download 260 genomes from NCBI

In [ ]:
import shutil, subprocess, zipfile
from pathlib import Path

accessions = ["GCA_050376075.1","GCA_045107525.1","GCA_021895815.1","GCA_045107315.1","GCA_050375315.1","GCA_050375435.1","GCA_050376755.1","GCA_026330395.1","GCA_056781265.1","GCA_021896375.1","GCA_021206595.1","GCA_021895555.1","GCA_021896795.1","GCA_050374115.1","GCA_050375415.1","GCA_050376315.1","GCA_050376735.1","GCA_021895935.1","GCA_050374815.1","GCA_050396405.1","GCA_045107215.1","GCA_014840495.1","GCA_050375795.1","GCA_052174395.1","GCA_021895735.1","GCA_050375995.1","GCA_050374935.1","GCA_021896145.1","GCA_050375235.1","GCA_050375175.1","GCA_021896335.1","GCA_050376095.1","GCA_050374695.1","GCA_021895615.1","GCA_021896435.1","GCA_014840645.1","GCA_021896235.1","GCA_021896315.1","GCA_050374715.1","GCA_050376175.1","GCA_050377715.1","GCA_021896015.1","GCA_050396445.1","GCA_045107695.1","GCA_030756695.1","GCA_050374595.1","GCA_021896095.1","GCA_050374535.1","GCA_021896355.1","GCA_021896735.1","GCA_050378275.1","GCA_050766175.1","GCA_021896775.1","GCA_045107225.1","GCA_021895675.1","GCA_022169645.1","GCA_021896245.1","GCA_050375115.1","GCA_014840475.1","GCA_021896595.1","GCA_021502955.1","GCA_056135545.1","GCA_050374495.1","GCA_050378135.1","GCA_014840575.1","GCA_050375275.1","GCA_050374675.1","GCA_050375935.1","GCA_026331055.1","GCA_050375815.1","GCA_003006175.1","GCA_021809275.1","GCA_050375125.1","GCA_050375645.1","GCA_018409235.1","GCA_050375375.1","GCA_014946945.1","GCA_021895895.1","GCA_021896195.1","GCA_050375835.1","GCA_021206755.1","GCA_014840555.1","GCA_021896475.1","GCA_021896115.1","GCA_030250545.1","GCA_021896135.1","GCA_059237735.1","GCA_021278205.1","GCA_021896455.1","GCA_021895795.1","GCA_050374575.1","GCA_045108045.1","GCA_050766155.1","GCA_021502915.1","GCA_045529805.1","GCA_052174375.1","GCA_050374895.1","GCA_014840405.1","GCA_021895855.1","GCA_014840305.1","GCA_019775735.1","GCA_045107715.1","GCA_014840605.1","GCA_050374875.1","GCA_045107825.1","GCA_014840535.1","GCA_050376005.1","GCA_045107625.1","GCA_050375845.1","GCA_050375355.1","GCA_026315245.1","GCA_050374615.1","GCA_050376635.1","GCA_021896515.1","GCA_050375955.1","GCA_021896755.1","GCA_030250445.1","GCA_021896495.1","GCA_050374095.1","GCA_021896615.1","GCA_050374475.1","GCA_021895575.1","GCA_021895635.1","GCA_045107455.1","GCA_026331495.1","GCA_050375705.1","GCA_050374655.1","GCA_050374515.1","GCA_021896575.1","GCA_021896555.1","GCA_050376775.1","GCA_021895655.1","GCA_030250415.1","GCA_030250465.1","GCA_021895915.1","GCA_026330515.1","GCA_018409345.1","GCA_045107955.1","GCA_021896035.1","GCA_021895595.1","GCA_021895835.1","GCA_014946925.1","GCA_021895975.1","GCA_045107235.1","GCA_050375555.1","GCA_021277345.1","GCA_050378035.1","GCA_021896415.1","GCA_021895955.1","GCA_050376815.1","GCA_050374635.1","GCA_021896955.1","GCA_050375635.1","GCA_021896215.1","GCA_026331815.1","GCA_052174695.1","GCA_021896655.1","GCA_050375895.1","GCA_050375155.1","GCA_050374975.1","GCA_021206715.1","GCA_030250505.1","GCA_014840345.1","GCA_021461325.1","GCA_050376255.1","GCA_050376285.1","GCA_050375915.1","GCA_045107765.1","GCA_021895535.1","GCA_014840415.1","GCA_021896715.1","GCA_014840595.1","GCA_021895715.1","GCA_050374835.1","GCA_014840245.1","GCA_021278145.1","GCA_021896275.1","GCA_050374995.1","GCA_021895755.1","GCA_026302175.1","GCA_045107865.1","GCA_045108035.1","GCA_026331935.1","GCA_021503035.1","GCA_050375095.1","GCA_050374755.1","GCA_021896395.1","GCA_018283475.1","GCA_021896175.1","GCA_021502995.1","GCA_045108095.1","GCA_050396465.1","GCA_030250345.1","GCA_014946935.1","GCA_050375675.1","GCA_050375495.1","GCA_050375465.1","GCA_026331915.1","GCA_045107805.1","GCA_030250525.1","GCA_050375945.1","GCA_050374735.1","GCA_026302215.1","GCA_050375875.1","GCA_045107565.1","GCA_014840315.1","GCA_050375245.1","GCA_026359975.1","GCA_050374855.1","GCA_021895875.1","GCA_050376955.1","GCA_026330955.1","GCA_021206735.1","GCA_014840395.1","GCA_021895775.1","GCA_021896295.1","GCA_014840295.1","GCA_045107935.1","GCA_045108065.1","GCA_021896055.1","GCA_021895995.1","GCA_045107485.1","GCA_045107285.1","GCA_050376115.1","GCA_021895695.1","GCA_050375615.1","GCA_014840325.1","GCA_021503115.1","GCA_055790525.1","GCA_050374915.1","GCA_050396485.1","GCA_050389635.1","GCA_021809345.1","GCA_018409155.1","GCA_045107245.1","GCA_030250565.1","GCA_021896635.1","GCA_056791625.1","GCA_021502945.1","GCA_050375035.1","GCA_014840635.1","GCA_045107345.1","GCA_050375455.1","GCA_050376795.1","GCA_050375075.1","GCA_014840435.1","GCA_050375015.1","GCA_050378295.1","GCA_045107395.1","GCA_021896075.1","GCA_050375535.1","GCA_050375395.1","GCA_050374795.1","GCA_050375215.1","GCA_030250385.1","GCA_021896695.1","GCA_045107655.1","GCA_050375195.1","GCA_014840505.1","GCA_021896535.1"]
print(len(accessions), 'accessions')

WORK = Path('/content/work'); ASM = WORK / 'assemblies'
ASM.mkdir(parents=True, exist_ok=True)
BATCH = 25
batches = [accessions[i:i+BATCH] for i in range(0, len(accessions), BATCH)]

for n, batch in enumerate(batches, 1):
    marker = WORK / f'.b{n}.done'
    if marker.exists():
        print(f'batch {n}/{len(batches)} cached'); continue
    lf = WORK / f'b{n}.txt'; lf.write_text('\n'.join(batch) + '\n')
    zp = WORK / f'b{n}.zip'
    res = subprocess.run(['datasets','download','genome','accession',
                          '--inputfile',str(lf),'--include','genome',
                          '--filename',str(zp),'--no-progressbar'],
                         capture_output=True, text=True)
    if res.returncode != 0 or not zp.exists():
        print(f'batch {n} FAILED:', res.stderr.strip()[:250]); continue
    with zipfile.ZipFile(zp) as zf:
        for m in zf.namelist():
            if not m.endswith(('.fna','.fa','.fasta')):
                continue
            parts = m.split('/')
            acc = parts[2] if len(parts) >= 3 else Path(m).stem
            with zf.open(m) as s, (ASM / f'{acc}.fna').open('wb') as d:
                shutil.copyfileobj(s, d)
    zp.unlink(); marker.touch()
    print(f'batch {n}/{len(batches)} ok ({len(list(ASM.glob("*.fna")))} genomes)')

fastas = sorted(ASM.glob('*.fna'))
print(f'\nDownloaded {len(fastas)}/{len(accessions)} '
      f'({sum(f.stat().st_size for f in fastas)/1e6:.0f} MB)')


## Cell 6 - Smoke test (finds the working CLI form)

In [ ]:
import subprocess
from pathlib import Path

TEST = WORK / 'smoke'; TEST.mkdir(exist_ok=True)
one = sorted(ASM.glob('*.fna'))[0]
print('test genome:', one.name)

variants = [
    ('v3 preset kpsc', ['kleborate','-a',str(one),'-o',str(TEST/'t1'),'-p','kpsc']),
    ('v3 preset kp',   ['kleborate','-a',str(one),'-o',str(TEST/'t2'),'-p','kp']),
    ('v3 module kpsc', ['kleborate','-a',str(one),'-o',str(TEST/'t3'),
                        '-m','klebsiella_pneumo_complex']),
    ('v2 --all',       ['kleborate','-a',str(one),'--all','-o',str(TEST/'t4.txt')]),
]

WORKING_CMD = None
for label, cmd in variants:
    print('--- trying:', label)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode == 0:
        print('    OK'); WORKING_CMD = label; break
    print('    rc=', r.returncode)
    print('    stderr:', (r.stderr or '').strip()[-700:] or '(none)')

if WORKING_CMD is None:
    raise SystemExit('All variants failed - send the output above.')
print('\n>>> using:', WORKING_CMD)

import pandas as pd
for p in sorted(TEST.rglob('*')):
    if p.is_file() and p.suffix in {'.txt','.tsv'} and p.stat().st_size > 0:
        df = pd.read_csv(p, sep='\t', dtype=str)
        print(p.name, df.shape); print('columns:', list(df.columns)); break


## Cell 7 - Full run

In [ ]:
import subprocess, time
from pathlib import Path

# Results live on Drive so a reclaimed runtime does not cost the whole run.
OUT = PERSIST / 'kleborate'; OUT.mkdir(parents=True, exist_ok=True)
CHUNK = 20

def build_cmd(chunk, tag):
    f = list(map(str, chunk))
    if WORKING_CMD == 'v3 preset kpsc':
        return ['kleborate','-a',*f,'-o',str(tag),'-p','kpsc']
    if WORKING_CMD == 'v3 preset kp':
        return ['kleborate','-a',*f,'-o',str(tag),'-p','kp']
    if WORKING_CMD == 'v3 module kpsc':
        return ['kleborate','-a',*f,'-o',str(tag),'-m','klebsiella_pneumo_complex']
    return ['kleborate','-a',*f,'--all','-o',str(tag)+'.txt']

# Process in the order of the (shuffled) accession list rather than
# sorted filename order. Accession numbers cluster by submitting centre
# and region, so a sorted run that stops early leaves a geographically
# skewed subset. In list order, any prefix is still a balanced random
# sample across groups - which matters because long runs get interrupted.
fastas = [ASM / f'{a}.fna' for a in accessions if (ASM / f'{a}.fna').exists()]
chunks = [fastas[i:i+CHUNK] for i in range(0, len(fastas), CHUNK)]
print(len(fastas), 'genomes in', len(chunks), 'chunks')

failed = []; t0 = time.time()
for n, chunk in enumerate(chunks, 1):
    tag = OUT / f'chunk_{n:03d}'; done = OUT / f'chunk_{n:03d}.done'
    if done.exists():
        print(f'chunk {n}/{len(chunks)} cached'); continue
    r = subprocess.run(build_cmd(chunk, tag), capture_output=True, text=True)
    if r.returncode != 0:
        failed.append(n)
        print(f'chunk {n} FAILED:', (r.stderr or '').strip()[-500:]); continue
    done.touch(); el = time.time() - t0
    print(f'chunk {n}/{len(chunks)} ok [{el/60:.1f} min, '
          f'~{(el/n)*(len(chunks)-n)/60:.1f} min left]')

print(f'\nTotal {(time.time()-t0)/60:.1f} min; failed: {failed}')


## Cell 8 - Collect, summarise, save

In [ ]:
import pandas as pd

# Kleborate v3 writes TWO tables per run:
#   klebsiella_pneumo_complex_output.txt               <- main, 1 row/genome
#   klebsiella_pneumo_complex_hAMRonization_output.txt <- AMR long format
# Concatenating both would interleave two different schemas, so take the
# main table only. The AMR table is collected separately below.
MAIN = 'klebsiella_pneumo_complex_output.txt'
AMRH = 'hAMRonization'

frames, amr_frames = [], []
for p in sorted(OUT.rglob('*.txt')):
    if not p.is_file() or p.stat().st_size == 0:
        continue
    try:
        df = pd.read_csv(p, sep='\t', dtype=str)
    except Exception as e:
        print('skip', p.name, e); continue
    if p.name == MAIN:
        frames.append(df)
    elif AMRH in p.name:
        amr_frames.append(df)

if not frames:
    raise SystemExit(f'No {MAIN} found - check the Cell 7 output.')

kleb = pd.concat(frames, ignore_index=True).drop_duplicates()
print('main table :', kleb.shape)
print('columns:', list(kleb.columns))

if amr_frames:
    amr = pd.concat(amr_frames, ignore_index=True).drop_duplicates()
    amr.to_csv('/content/kleborate_pakistan_kp_amr.csv', index=False)
    amr.to_csv(PERSIST / 'kleborate_pakistan_kp_amr.csv', index=False)
    print('AMR table  :', amr.shape, '-> kleborate_pakistan_kp_amr.csv')

def pick(df, *cands):
    low = {c.lower().replace(' ','_'): c for c in df.columns}
    for c in cands:
        if c in df.columns: return c
        k = c.lower().replace(' ','_')
        if k in low: return low[k]
    return None

col_strain = pick(kleb,'strain','Genome Name','Name','assembly')
col_k      = pick(kleb,'K_locus','K locus','Best match locus','K_type')
col_kconf  = pick(kleb,'K_locus_confidence','K locus confidence','Match confidence')
col_o      = pick(kleb,'O_locus','O locus','O_type')
col_st     = pick(kleb,'ST','MLST ST','st')
print('resolved:', col_strain, col_k, col_kconf, col_o, col_st)

if col_k:
    kd = kleb[col_k].fillna('unknown').value_counts()
    print(f'\n=== K-locus distribution ({kd.size} types) ===')
    print(kd.head(30).to_string())
    print(f'\nTop 10 cover {kd.head(10).sum()/kd.sum()*100:.1f}%')
for lab, c in [('K confidence',col_kconf),('O-locus',col_o),('ST',col_st)]:
    if c:
        print(f'\n=== {lab} ===')
        print(kleb[c].fillna('unknown').value_counts().head(15).to_string())

if col_strain:
    kleb['assembly'] = (kleb[col_strain].astype(str)
                        .str.replace(r'\.(fna|fa|fasta)$','',regex=True).str.strip())

# Write to Drive as well as local scratch, so the result survives even if
# the browser download in the next cell is missed.
kleb.to_csv('/content/kleborate_pakistan_kp.csv', index=False)
kleb.to_csv(PERSIST / 'kleborate_pakistan_kp.csv', index=False)
print('\nwrote kleborate_pakistan_kp.csv', kleb.shape)
print('  -> /content/ and', PERSIST)


## Cell 9 - Download the result

In [ ]:
from google.colab import files
files.download('/content/kleborate_pakistan_kp.csv')
